In [73]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from catboost import CatBoostRegressor


import sklearn
sklearn.set_config(transform_output="pandas")
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_squared_log_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler, MinMaxScaler, OrdinalEncoder, TargetEncoder, FunctionTransformer

In [74]:
df = pd.read_csv('../train.csv')

In [75]:
df = pd.read_csv('../train.csv')
df=df.drop_duplicates()
df = df.drop('Id', axis=1, errors='ignore') #убран из клинера
condition = (df['GrLivArea'] > 4000) & (df['SalePrice'] < 160001)
df=df[~condition]

Q1_price = df['SalePrice'].quantile(0.25) #очистка выбросов по цене
Q3_price = df['SalePrice'].quantile(0.75)
IQR_price = Q3_price - Q1_price
lower_bound_price = Q1_price - 1.5 * IQR_price
upper_bound_price = Q3_price + 1.5 * IQR_price
    
condition_price = (df['SalePrice'] < lower_bound_price) | (df['SalePrice'] > upper_bound_price)
df = df[~condition_price]

Q1_lot = df['LotArea'].quantile(0.25) #очистка выбросов по площади
Q3_lot = df['LotArea'].quantile(0.75)
IQR_lot = Q3_lot - Q1_lot
upper_bound_lot = Q3_lot + 3 * IQR_lot

condition_lot = df['LotArea'] > upper_bound_lot
df = df[~condition_lot]


cond_frontage = df['LotFrontage'] > 250 #LotFrontage - по рекомендации > 250 
df = df[~cond_frontage]

df['TotalPorchSF'] = (df['OpenPorchSF'] + df['EnclosedPorch'] + #Веранды
                          df['3SsnPorch'] + df['ScreenPorch'])
cond_porch = df['TotalPorchSF'] > 700
df = df[~cond_porch]
df = df.drop('TotalPorchSF', axis=1)
    

In [76]:
X, y = df.drop('SalePrice', axis=1), df['SalePrice']

y_log = np.log1p(y)

X_train, X_valid, y_train_log, y_valid_log = train_test_split(X, y_log, test_size=0.3, random_state=42)

y_train_original = np.expm1(y_train_log)
y_valid_original = np.expm1(y_valid_log)

In [77]:
cat_features = ['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
                'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
                'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
                'Exterior2nd', 'MasVnrType', 'Foundation', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
                'Heating', 'CentralAir', 'Electrical', 'Functional', 'GarageType', 'GarageFinish',
                'PavedDrive', 'Fence', 'MiscFeature','SaleType', 'SaleCondition', 'PoolQC', 'ExterQual','ExterCond',
                'BsmtQual', 'BsmtCond','HeatingQC','KitchenQual','FireplaceQu', 'GarageQual', 'GarageCond']
num_features = ['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
                'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1',
                'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
                'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
                'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd',
                'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF',
                'OpenPorchSF', 'EnclosedPorch', 'ScreenPorch', 'PoolArea', 'MoSold', 'YrSold', '3SsnPorch', 'MiscVal' ]

In [78]:
# def clean_data(X):
#     X['MasVnrType'] = X['MasVnrType'].fillna('None')
#     X['FireplaceQu'] = X['FireplaceQu'].fillna('None')
#     X['GarageQual'] = X['GarageQual'].fillna('None')
#     X['GarageFinish'] = X['GarageFinish'].fillna('None')
#     X['GarageType'] = X['GarageType'].fillna('None')
#     X['GarageCond'] = X['GarageCond'].fillna('None')
#     X['GarageAge'] = 2026 - X['GarageYrBlt']
#     X['RemAge'] = 2026 - X['YearRemodAdd']
#     X['HouseAge'] = 2026 - X['YearBuilt']
#     X['IsNew'] = (X['HouseAge'] <= 5).astype(int)
#     X['IsOld'] = (X['HouseAge'] >= 70).astype(int)
#     X['IsHistoric'] = (X['HouseAge'] >= 100).astype(int)

#     X['TotalSF'] = (X['GrLivArea'] + 
#                     X['TotalBsmtSF'].fillna(0) + 
#                     X['GarageArea'].fillna(0) +
#                     X['WoodDeckSF'] + 
#                     X['OpenPorchSF'] + 
#                     X['EnclosedPorch'] + 
#                     X['3SsnPorch'] + 
#                     X['ScreenPorch'])
#     return X

# def imputer_groupby_Neighborhood(X):
#     X = X.copy()
#     X['LotFrontage'] = X.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))
#     return X

# cleaner = FunctionTransformer(clean_data)
# group_imputer = FunctionTransformer(imputer_groupby_Neighborhood)

# drop_features = ['Id','PoolQC', 'Alley', 'MiscFeature']

# # FireplaceQu_order = ['Ex', 'Gd', 'TA', 'Fa', 'Po']

# my_imputer = ColumnTransformer(
#     transformers = [
#         ('drop_features', 'drop', drop_features),
#         ('num', Pipeline([
#             ('imputer', SimpleImputer(strategy='median')),
#             ('scaler', StandardScaler())
#         ]), num_features),
#         ('cat', Pipeline([
#             ('imputer', SimpleImputer(strategy='most_frequent')),
#             ('encoder', TargetEncoder(smooth=10, target_type='continuous'))
#         ]), cat_features)
#     ],
#     verbose_feature_names_out = False,
#     remainder = 'passthrough'
# )


# preprocessor = Pipeline(
#     [
#         ('cleaner', cleaner),
#         ('group_imputer', group_imputer),
#         ('imputer', my_imputer)
#     ]
# )

In [79]:
def clean_data(X):
    X=X.copy()
    X['MasVnrType'] = X['MasVnrType'].fillna('None')
    X['FireplaceQu'] = X['FireplaceQu'].fillna('None')
    X['GarageQual'] = X['GarageQual'].fillna('None')
    X['GarageFinish'] = X['GarageFinish'].fillna('None')
    X['GarageType'] = X['GarageType'].fillna('None')
    X['GarageCond'] = X['GarageCond'].fillna('None')

    X['Alley'] = X['Alley'].fillna('None') #добавлена обработка
    X['PoolQC'] = X['PoolQC'].fillna('None') #добавлена обработка
    X['GarageType'] = X['GarageType'].fillna('None') #добавлена обработка

    X['GarageArea'] = X['GarageArea'].fillna(0) #добавлена обработка 
    X['MasVnrArea'] = X['MasVnrArea'].fillna(0) #добавлена обработка 

    X['GarageYrBlt'] = X['GarageYrBlt'].fillna(X['YearBuilt'])
    X['GarageAge'] = X['YrSold'] - X['GarageYrBlt']
    X['RemAge'] = X['YrSold'] - X['YearRemodAdd']
    X['HouseAge'] = X['YrSold'] - X['YearBuilt']
    X['IsNew'] = (X['HouseAge'] <= 5).astype(int)
    X['IsOld'] = (X['HouseAge'] >= 70).astype(int)
    X['IsHistoric'] = (X['HouseAge'] >= 100).astype(int)
    X['TotalBath'] = (X['FullBath'] + (0.5 * X['HalfBath']) +  #добавлено
                      X['BsmtFullBath'] + (0.5 * X['BsmtHalfBath']))
    X['HasPool'] = X['PoolArea'].apply(lambda x: 1 if x > 0 else 0)
    X['TotalSF'] = (X['GrLivArea'] + 
                    X['TotalBsmtSF'].fillna(0) + 
                    X['GarageArea'].fillna(0) +
                    X['WoodDeckSF'] + 
                    X['OpenPorchSF'] + 
                    X['EnclosedPorch'] + 
                    X['3SsnPorch'] + 
                    X['ScreenPorch'])
    
    X['TotalBathrooms'] = X['FullBath'] + 0.5*X['HalfBath'] + X['BsmtFullBath'] + 0.5*X['BsmtHalfBath']
    X['TotalRooms'] = X['TotRmsAbvGrd'] + X['BedroomAbvGr']
    X['AreaPerRoom'] = X['GrLivArea'] / (X['TotRmsAbvGrd'] + 1)
    X['QualityScore'] = X['OverallQual'] * X['OverallCond']
    X['IsRenovated'] = (X['YearRemodAdd'] != X['YearBuilt']).astype(int)
    X['YearsSinceRenovation'] = X['YrSold'] - X['YearRemodAdd']
    
    return X

new_num_features = num_features + ['GarageAge', 'RemAge', 'HouseAge', 'TotalSF', 'TotalBath', 
                                   'IsNew', 'IsOld', 'IsHistoric', 'HasPool']

def imputer_groupby_Neighborhood(X):
    X = X.copy()
    X['LotFrontage'] = X.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))
    return X

cleaner = FunctionTransformer(clean_data)
group_imputer = FunctionTransformer(imputer_groupby_Neighborhood)

drop_features = ['MiscFeature', 'Utilities', 'PoolQC', 'Alley', 'Fence',
                 'LowQualFinSF', 'MiscVal', 'HalfBath', 'BsmtHalfBath', 'Street', 'LandSlope'] 

my_imputer = ColumnTransformer(
    transformers = [
        ('drop_features', 'drop', drop_features),
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', RobustScaler())
        ]), new_num_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_features)
    ],
    verbose_feature_names_out=False,
    remainder='passthrough'
)

preprocessor = Pipeline(
    [
        ('cleaner', cleaner),
        ('group_imputer', group_imputer),
        ('imputer', my_imputer)
    ]
)

In [80]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error, mean_squared_error
from sklearn.feature_selection import SelectFromModel
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("ОБУЧЕНИЕ С Lasso ДЛЯ ОТБОРА ПРИЗНАКОВ")
print("="*60)

# Подготовка данных
X_train, X_valid, y_train_log, y_valid_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

# Обучаем препроцессор
print("\n1. Применяем препроцессинг...")
preprocessor.fit(X_train, y_train_log)

# Преобразуем данные
X_train_processed = preprocessor.transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)

# Находим индексы категориальных колонок
if hasattr(X_train_processed, 'columns'):
    cat_indices = [i for i, col in enumerate(X_train_processed.columns) 
                   if X_train_processed[col].dtype == 'object']
    X_train_array = X_train_processed.values
    X_valid_array = X_valid_processed.values
    feature_names = X_train_processed.columns.tolist()
else:
    X_train_array = X_train_processed
    X_valid_array = X_valid_processed
    feature_names = [f'feature_{i}' for i in range(X_train_array.shape[1])]

print(f"Исходное количество признаков: {X_train_array.shape[1]}")

# ============================================
# Lasso для отбора признаков
# ============================================
print("\n2. Отбор признаков с помощью Lasso...")

# Стандартизируем данные для Lasso
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_array)
X_valid_scaled = scaler.transform(X_valid_array)

# Подбираем оптимальный alpha для Lasso
from sklearn.linear_model import LassoCV

print("   Поиск оптимального alpha...")
lasso_cv = LassoCV(
    cv=5, 
    random_state=42, 
    alphas=np.logspace(-4, 0, 50),
    max_iter=10000
)
lasso_cv.fit(X_train_scaled, y_train_log.values)

print(f"   Лучший alpha: {lasso_cv.alpha_:.6f}")
print(f"   Количество ненулевых коэффициентов: {np.sum(lasso_cv.coef_ != 0)}")

# Отбираем признаки
selector = SelectFromModel(lasso_cv, threshold=1e-5, prefit=True)
X_train_selected = selector.transform(X_train_scaled)
X_valid_selected = selector.transform(X_valid_scaled)

selected_features = [feature_names[i] for i in range(len(feature_names)) 
                     if selector.get_support()[i]]

print(f"   Отобрано признаков: {len(selected_features)} из {len(feature_names)}")
print(f"   Процент отбора: {len(selected_features)/len(feature_names)*100:.1f}%")

# ============================================
# Обучение CatBoost на отобранных признаках
# ============================================
print("\n3. Обучение CatBoost на отобранных признаках...")

# Находим индексы категориальных признаков в отобранных
selected_cat_indices = []
for i, feat in enumerate(selected_features):
    if feat in cat_features:
        selected_cat_indices.append(i)

print(f"   Категориальных признаков в отобранных: {len(selected_cat_indices)}")

# Параметры CatBoost
cb_params = {
    'bootstrap_type': 'Bayesian',
    'iterations': 900,
    'learning_rate': 0.0279,
    'depth': 3,
    'l2_leaf_reg': 12.1,
    'bagging_temperature': 5.06,
    'random_strength': 7.64,
    'colsample_bylevel': 0.49,
    'min_data_in_leaf': 225,
    'eval_metric': 'RMSE',
    'loss_function': 'RMSE',
    'early_stopping_rounds': 44,
    'max_bin': 128,
    'verbose': 100,
    'random_seed': 42,
    'cat_features': selected_cat_indices
}

cb = CatBoostRegressor(**cb_params)

# Обучаем
cb.fit(
    X_train_selected,
    y_train_log.values,
    eval_set=(X_valid_selected, y_valid_log.values),
    plot=True
)

# Оценка
y_pred_train = cb.predict(X_train_selected)
y_pred_valid = cb.predict(X_valid_selected)

train_rmsle = np.sqrt(mean_squared_log_error(
    np.expm1(y_train_log.values), 
    np.expm1(y_pred_train)
))
valid_rmsle = np.sqrt(mean_squared_log_error(
    np.expm1(y_valid_log.values), 
    np.expm1(y_pred_valid)
))

train_rmse = np.sqrt(mean_squared_error(y_train_log.values, y_pred_train))
valid_rmse = np.sqrt(mean_squared_error(y_valid_log.values, y_pred_valid))

print("\n" + "="*60)
print("РЕЗУЛЬТАТЫ ПОСЛЕ LASSO")
print("="*60)
print(f"Train RMSLE: {train_rmsle:.6f}")
print(f"Valid RMSLE: {valid_rmsle:.6f}")
print(f"Разница: {((valid_rmsle - train_rmsle) / train_rmsle * 100):.2f}%")
print(f"\nTrain RMSE: ${np.expm1(train_rmse):.2f}")
print(f"Valid RMSE: ${np.expm1(valid_rmse):.2f}")

# ============================================
# Создание сабмишна с Lasso
# ============================================
print("\n" + "="*60)
print("СОЗДАНИЕ САБМИШНА С LASSO")
print("="*60)

# Обучаем на ВСЕХ данных с отбором признаков
print("Обучение финальной модели на всех данных...")

# Преобразуем все данные
X_all_processed = preprocessor.transform(X)
X_all_scaled = scaler.fit_transform(X_all_processed.values if hasattr(X_all_processed, 'values') else X_all_processed)
X_all_selected = selector.transform(X_all_scaled)

# Обучаем финальную модель
final_model = CatBoostRegressor(**cb_params)
final_model.fit(X_all_selected, y_log.values, verbose=False)

# Предсказание
test = pd.read_csv('../test.csv')
test_ids = test['Id'].copy()
test_processed = preprocessor.transform(test)
test_scaled = scaler.transform(test_processed.values if hasattr(test_processed, 'values') else test_processed)
test_selected = selector.transform(test_scaled)

test_pred_log = final_model.predict(test_selected)
test_pred = np.expm1(test_pred_log)

# Сохраняем
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': test_pred
})
submission.to_csv('submission_lasso.csv', index=False)

print(f"\n✅ Сабмишн сохранен в 'submission_lasso.csv'")
print(f"📊 Ожидаемый публичный RMSLE: ~{valid_rmsle + 0.008:.4f}")

# Важность признаков после Lasso
print("\n" + "="*60)
print("ТОП-15 ВАЖНЫХ ПРИЗНАКОВ (после Lasso)")
print("="*60)

importance_df = pd.DataFrame({
    'feature': selected_features,
    'importance': cb.feature_importances_
}).sort_values('importance', ascending=False).head(15)

for idx, row in importance_df.iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f}")

ОБУЧЕНИЕ С Lasso ДЛЯ ОТБОРА ПРИЗНАКОВ

1. Применяем препроцессинг...
Исходное количество признаков: 301

2. Отбор признаков с помощью Lasso...
   Поиск оптимального alpha...
   Лучший alpha: 0.004292
   Количество ненулевых коэффициентов: 95
   Отобрано признаков: 91 из 301
   Процент отбора: 30.2%

3. Обучение CatBoost на отобранных признаках...
   Категориальных признаков в отобранных: 0


MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

0:	learn: 0.3477128	test: 0.3594810	best: 0.3594810 (0)	total: 637us	remaining: 574ms
100:	learn: 0.1564206	test: 0.1702836	best: 0.1702836 (100)	total: 71.1ms	remaining: 563ms
200:	learn: 0.1244572	test: 0.1361156	best: 0.1361156 (200)	total: 145ms	remaining: 505ms
300:	learn: 0.1140595	test: 0.1246088	best: 0.1246088 (300)	total: 232ms	remaining: 461ms
400:	learn: 0.1094674	test: 0.1201492	best: 0.1201492 (400)	total: 315ms	remaining: 392ms
500:	learn: 0.1054941	test: 0.1167467	best: 0.1167467 (500)	total: 420ms	remaining: 334ms
600:	learn: 0.1025637	test: 0.1145468	best: 0.1145468 (600)	total: 510ms	remaining: 254ms
700:	learn: 0.1001636	test: 0.1131059	best: 0.1131059 (700)	total: 582ms	remaining: 165ms
800:	learn: 0.0978747	test: 0.1119160	best: 0.1119159 (799)	total: 684ms	remaining: 84.6ms
899:	learn: 0.0961385	test: 0.1110682	best: 0.1110682 (899)	total: 765ms	remaining: 0us

bestTest = 0.1110681967
bestIteration = 899


РЕЗУЛЬТАТЫ ПОСЛЕ LASSO
Train RMSLE: 0.096138
Valid RMSLE:

In [82]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("ПОИСК ОПТИМАЛЬНОГО ALPHA ДЛЯ LASSO (УПРОЩЕННАЯ ВЕРСИЯ)")
print("="*80)

# Подготовка данных
X_train, X_valid, y_train_log, y_valid_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

# Обучаем препроцессор
preprocessor.fit(X_train, y_train_log)

# Преобразуем данные и сразу конвертируем в numpy
X_train_processed = preprocessor.transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)

# Принудительно конвертируем в numpy
if isinstance(X_train_processed, pd.DataFrame):
    X_train_array = X_train_processed.to_numpy()
    X_valid_array = X_valid_processed.to_numpy()
else:
    X_train_array = np.array(X_train_processed)
    X_valid_array = np.array(X_valid_processed)

print(f"Размер данных: {X_train_array.shape}")
print(f"Тип данных: {X_train_array.dtype}")

# Определяем числовые и категориальные колонки по типу данных
# Пробуем преобразовать первые несколько строк в числа
numeric_mask = []
for i in range(X_train_array.shape[1]):
    try:
        # Пробуем преобразовать первые 10 значений в float
        sample = X_train_array[:10, i]
        # Проверяем, можно ли преобразовать в числа
        float(sample[0])
        # Если успешно - это числовая колонка
        numeric_mask.append(True)
    except (ValueError, TypeError):
        # Если не получается - это категориальная
        numeric_mask.append(False)

numeric_indices = np.where(numeric_mask)[0]
categorical_indices = np.where(~np.array(numeric_mask))[0]

print(f"Числовых признаков: {len(numeric_indices)}")
print(f"Категориальных признаков: {len(categorical_indices)}")

# Извлекаем данные
X_train_numeric = X_train_array[:, numeric_indices].astype(float)
X_valid_numeric = X_valid_array[:, numeric_indices].astype(float)

X_train_categorical = X_train_array[:, categorical_indices]
X_valid_categorical = X_valid_array[:, categorical_indices]

# Стандартизируем числовые данные
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_numeric)
X_valid_scaled = scaler.transform(X_valid_numeric)

# ============================================
# Тестируем разные alpha
# ============================================
alphas_to_test = [0.0005, 0.001, 0.002, 0.005, 0.008, 0.01, 0.02, 0.05]

results = []

print("\n" + "="*80)
print("ТЕСТИРОВАНИЕ РАЗНЫХ ALPHA")
print("="*80)

for alpha in alphas_to_test:
    print(f"\nТестируем alpha = {alpha}")
    
    # Lasso
    lasso = Lasso(alpha=alpha, random_state=42, max_iter=10000)
    lasso.fit(X_train_scaled, y_train_log.values)
    
    # Количество отобранных признаков
    selected = np.abs(lasso.coef_) > 1e-5
    n_selected = np.sum(selected)
    print(f"  Отобрано признаков: {n_selected} из {X_train_scaled.shape[1]}")
    
    if n_selected < 10:
        print(f"  ⚠️ Слишком мало признаков, пропускаем")
        results.append({
            'alpha': alpha,
            'n_features': n_selected,
            'valid_rmsle': None
        })
        continue
    
    # Отбираем признаки
    X_train_selected = X_train_scaled[:, selected]
    X_valid_selected = X_valid_scaled[:, selected]
    
    # Объединяем с категориальными
    X_train_final = np.hstack([X_train_selected, X_train_categorical])
    X_valid_final = np.hstack([X_valid_selected, X_valid_categorical])
    
    # Индексы категориальных признаков
    cat_indices = list(range(X_train_selected.shape[1], X_train_final.shape[1]))
    
    # Обучаем CatBoost
    cb = CatBoostRegressor(
        bootstrap_type='Bayesian',
        iterations=600,
        learning_rate=0.03,
        depth=3,
        l2_leaf_reg=12,
        bagging_temperature=5,
        random_strength=7,
        colsample_bylevel=0.5,
        min_data_in_leaf=200,
        eval_metric='RMSE',
        loss_function='RMSE',
        early_stopping_rounds=35,
        max_bin=128,
        cat_features=cat_indices,
        verbose=False,
        random_seed=42
    )
    
    cb.fit(X_train_final, y_train_log.values, 
           eval_set=(X_valid_final, y_valid_log.values), 
           verbose=False)
    
    # Оценка
    y_pred_valid = cb.predict(X_valid_final)
    valid_rmsle = np.sqrt(mean_squared_log_error(
        np.expm1(y_valid_log.values), 
        np.expm1(y_pred_valid)
    ))
    
    y_pred_train = cb.predict(X_train_final)
    train_rmsle = np.sqrt(mean_squared_log_error(
        np.expm1(y_train_log.values), 
        np.expm1(y_pred_train)
    ))
    
    diff = (valid_rmsle - train_rmsle) / train_rmsle * 100
    
    print(f"  Train RMSLE: {train_rmsle:.6f}")
    print(f"  Valid RMSLE: {valid_rmsle:.6f}")
    print(f"  Разница: {diff:.2f}%")
    
    results.append({
        'alpha': alpha,
        'n_features': n_selected,
        'train_rmsle': train_rmsle,
        'valid_rmsle': valid_rmsle,
        'diff': diff
    })

# ============================================
# Вывод результатов
# ============================================
print("\n" + "="*80)
print("СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
print("="*80)

for r in results:
    if r['valid_rmsle'] is not None:
        print(f"alpha={r['alpha']:.4f}: {r['n_features']} признаков, Valid RMSLE={r['valid_rmsle']:.6f}, Разница={r['diff']:.2f}%")

# Находим лучший alpha
best_result = None
for r in results:
    if r['valid_rmsle'] is not None:
        if best_result is None or r['valid_rmsle'] < best_result['valid_rmsle']:
            best_result = r

if best_result:
    print("\n" + "="*80)
    print(f"🏆 ЛУЧШИЙ ALPHA: {best_result['alpha']}")
    print("="*80)
    print(f"Valid RMSLE: {best_result['valid_rmsle']:.6f}")
    print(f"Train RMSLE: {best_result['train_rmsle']:.6f}")
    print(f"Разница: {best_result['diff']:.2f}%")
    print(f"Отобрано признаков: {best_result['n_features']}")
    
    # ============================================
    # Обучение финальной модели с лучшим alpha
    # ============================================
    print("\n" + "="*80)
    print("ОБУЧЕНИЕ ФИНАЛЬНОЙ МОДЕЛИ")
    print("="*80)
    
    best_alpha = best_result['alpha']
    
    # Обучаем Lasso на всех данных
    lasso_best = Lasso(alpha=best_alpha, random_state=42, max_iter=10000)
    lasso_best.fit(X_train_scaled, y_train_log.values)
    
    selected_best = np.abs(lasso_best.coef_) > 1e-5
    
    # Подготавливаем финальные данные
    X_train_selected_best = X_train_scaled[:, selected_best]
    X_valid_selected_best = X_valid_scaled[:, selected_best]
    
    X_train_final_best = np.hstack([X_train_selected_best, X_train_categorical])
    X_valid_final_best = np.hstack([X_valid_selected_best, X_valid_categorical])
    
    cat_indices_best = list(range(X_train_selected_best.shape[1], X_train_final_best.shape[1]))
    
    # Обучаем CatBoost
    cb_best = CatBoostRegressor(
        bootstrap_type='Bayesian',
        iterations=900,
        learning_rate=0.0279,
        depth=3,
        l2_leaf_reg=12.1,
        bagging_temperature=5.06,
        random_strength=7.64,
        colsample_bylevel=0.49,
        min_data_in_leaf=225,
        eval_metric='RMSE',
        loss_function='RMSE',
        early_stopping_rounds=44,
        max_bin=128,
        cat_features=cat_indices_best,
        verbose=100,
        random_seed=42
    )
    
    cb_best.fit(
        X_train_final_best,
        y_train_log.values,
        eval_set=(X_valid_final_best, y_valid_log.values),
        plot=True
    )
    
    # Финальная оценка
    y_pred_valid = cb_best.predict(X_valid_final_best)
    valid_rmsle_final = np.sqrt(mean_squared_log_error(
        np.expm1(y_valid_log.values), 
        np.expm1(y_pred_valid)
    ))
    
    print("\n" + "="*80)
    print("ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ")
    print("="*80)
    print(f"Valid RMSLE: {valid_rmsle_final:.6f}")
    
    # Создание сабмишна
    print("\nСоздание сабмишна...")
    
    # Подготавливаем все данные
    X_all_processed = preprocessor.transform(X)
    if isinstance(X_all_processed, pd.DataFrame):
        X_all_array = X_all_processed.to_numpy()
    else:
        X_all_array = np.array(X_all_processed)
    
    X_all_numeric = X_all_array[:, numeric_indices].astype(float)
    X_all_categorical = X_all_array[:, categorical_indices]
    
    X_all_scaled = scaler.transform(X_all_numeric)
    X_all_selected = X_all_scaled[:, selected_best]
    
    X_all_final = np.hstack([X_all_selected, X_all_categorical])
    
    # Обучаем финальную модель
    final_model = CatBoostRegressor(
        bootstrap_type='Bayesian',
        iterations=900,
        learning_rate=0.0279,
        depth=3,
        l2_leaf_reg=12.1,
        bagging_temperature=5.06,
        random_strength=7.64,
        colsample_bylevel=0.49,
        min_data_in_leaf=225,
        eval_metric='RMSE',
        loss_function='RMSE',
        early_stopping_rounds=44,
        max_bin=128,
        cat_features=cat_indices_best,
        verbose=False,
        random_seed=42
    )
    
    final_model.fit(X_all_final, y_log.values, verbose=False)
    
    # Предсказание
    test = pd.read_csv('../test.csv')
    test_ids = test['Id'].copy()
    test_processed = preprocessor.transform(test)
    
    if isinstance(test_processed, pd.DataFrame):
        test_array = test_processed.to_numpy()
    else:
        test_array = np.array(test_processed)
    
    test_numeric = test_array[:, numeric_indices].astype(float)
    test_categorical = test_array[:, categorical_indices]
    
    test_scaled = scaler.transform(test_numeric)
    test_selected = test_scaled[:, selected_best]
    
    test_final = np.hstack([test_selected, test_categorical])
    
    test_pred_log = final_model.predict(test_final)
    test_pred = np.expm1(test_pred_log)
    
    submission = pd.DataFrame({
        'Id': test_ids,
        'SalePrice': test_pred
    })
    submission.to_csv('submission_lasso_optimized.csv', index=False)
    
    print(f"\n✅ Сабмишн сохранен в 'submission_lasso_optimized.csv'")
    print(f"📊 Ожидаемый публичный RMSLE: ~{valid_rmsle_final + 0.008:.4f}")

else:
    print("⚠️ Нет успешных результатов для выбора alpha")

ПОИСК ОПТИМАЛЬНОГО ALPHA ДЛЯ LASSO (УПРОЩЕННАЯ ВЕРСИЯ)
Размер данных: (1092, 301)
Тип данных: float64
Числовых признаков: 301
Категориальных признаков: 0

ТЕСТИРОВАНИЕ РАЗНЫХ ALPHA

Тестируем alpha = 0.0005
  Отобрано признаков: 199 из 301


InvalidIndexError: (slice(None, None, None), array([False,  True,  True,  True,  True,  True,  True,  True,  True,
        True, False,  True, False, False, False, False,  True, False,
       False, False, False,  True,  True,  True, False,  True, False,
        True, False, False,  True,  True,  True, False, False, False,
       False,  True,  True,  True,  True, False,  True,  True, False,
        True,  True, False, False,  True,  True, False, False,  True,
        True, False,  True,  True, False, False,  True,  True, False,
        True, False,  True,  True,  True,  True, False, False,  True,
        True,  True,  True,  True,  True,  True, False,  True,  True,
        True, False,  True,  True,  True,  True,  True,  True,  True,
        True, False,  True, False,  True,  True,  True,  True,  True,
       False,  True,  True,  True,  True,  True,  True,  True,  True,
       False, False,  True,  True, False,  True,  True,  True,  True,
       False,  True,  True, False,  True, Fal

In [ ]:
preprocessor.fit(X_train, y_train_log)
preprocessed_X_train = preprocessor.transform(X_train)
preprocessed_X_valid = preprocessor.transform(X_valid)

In [ ]:
# cb = CatBoostRegressor(
#     cat_features = cat_features,
#     bootstrap_type = 'Bayesian',
#     iterations=1050,
#     learning_rate=0.029488648225486247,
#     depth=4,
#     l2_leaf_reg=9.102682068690584,
#     bagging_temperature=5.3167343078899725,
#     random_strength=4.341398861055912,
#     colsample_bylevel=0.6973773873201035,
#     min_data_in_leaf = 220,
#     eval_metric='RMSE',
#     loss_function = 'RMSE',
#     early_stopping_rounds=50,
#     max_bin = 128,
#     verbose=100,
#     random_seed=42
# )

# Умеренное усиление регуляризации
# Лучшие параметры (возврат к предыдущим)
cb = CatBoostRegressor(
    cat_features=cat_features,
    bootstrap_type='Bayesian',
    iterations=900,
    learning_rate=0.0279,
    depth=3,
    l2_leaf_reg=12.1,
    bagging_temperature=5.06,
    random_strength=7.64,
    colsample_bylevel=0.49,
    min_data_in_leaf=225,
    eval_metric='RMSE',
    loss_function='RMSE',
    early_stopping_rounds=44,
    max_bin=128,
    verbose=100,
    random_seed=42
)

cb.fit(
    preprocessed_X_train,
    y_train_log,
    eval_set=(preprocessed_X_valid, y_valid_log),
    plot=True
)



ValueError: list.index(x): x not in list

In [ ]:
y_pred_train_log = cb.predict(preprocessed_X_train)
y_pred_valid_log = cb.predict(preprocessed_X_valid)

# Возвращаем к исходному масштабу
y_pred_train = np.expm1(y_pred_train_log)
y_pred_valid = np.expm1(y_pred_valid_log)

In [ ]:
train_rmse = np.sqrt(mean_squared_error(y_train_original, y_pred_train))
train_mae = mean_absolute_error(y_train_original, y_pred_train)
train_r2 = r2_score(y_train_original, y_pred_train)

# Метрики для validation
valid_rmse = np.sqrt(mean_squared_error(y_valid_original, y_pred_valid))
valid_mae = mean_absolute_error(y_valid_original, y_pred_valid)
valid_r2 = r2_score(y_valid_original, y_pred_valid)

print("Train Metrics:")
print(f"RMSE: {train_rmse:.4f}")
print(f"MAE: {train_mae:.4f}")
print(f"R²: {train_r2:.4f}")
print("\nValidation Metrics:")
print(f"RMSE: {valid_rmse:.4f}")
print(f"MAE: {valid_mae:.4f}")
print(f"R²: {valid_r2:.4f}")

Train Metrics:
RMSE: 16374.7465
MAE: 11679.5132
R²: 0.9238

Validation Metrics:
RMSE: 18652.0228
MAE: 12888.8161
R²: 0.8971


In [ ]:
from sklearn.metrics import mean_squared_log_error, mean_squared_error, mean_absolute_error, r2_score

# Предсказания
y_pred_train_log = cb.predict(preprocessed_X_train)
y_pred_valid_log = cb.predict(preprocessed_X_valid)

# Метрики на логарифмической шкале (для RMSLE нужно expm1)
train_rmsle = np.sqrt(mean_squared_log_error(
    np.expm1(y_train_log.values), 
    np.expm1(y_pred_train_log)
))
valid_rmsle = np.sqrt(mean_squared_log_error(
    np.expm1(y_valid_log.values), 
    np.expm1(y_pred_valid_log)
))

# Дополнительные метрики для понимания
train_rmse_orig = np.sqrt(mean_squared_error(
    np.expm1(y_train_log.values), 
    np.expm1(y_pred_train_log)
))
valid_rmse_orig = np.sqrt(mean_squared_error(
    np.expm1(y_valid_log.values), 
    np.expm1(y_pred_valid_log)
))

print("="*60)
print("МЕТРИКИ ДЛЯ СОРЕВНОВАНИЯ")
print("="*60)
print(f"Train RMSLE: {train_rmsle:.6f}")
print(f"Valid RMSLE: {valid_rmsle:.6f}")
print(f"Разница RMSLE: {((valid_rmsle - train_rmsle) / train_rmsle * 100):.2f}%")
print(f"\nRMSE (ориг. шкала):")
print(f"Train: ${train_rmse_orig:.2f}")
print(f"Valid: ${valid_rmse_orig:.2f}")

МЕТРИКИ ДЛЯ СОРЕВНОВАНИЯ
Train RMSLE: 0.101052
Valid RMSLE: 0.127089
Разница RMSLE: 25.77%

RMSE (ориг. шкала):
Train: $16374.75
Valid: $18652.02


In [ ]:
# final_pipeline = Pipeline([('preprocessor', preprocessor), ('CatBoost', cb)])
# final_pipeline.fit(X_train, y_train_log)
# test = pd.read_csv('../test.csv')
# result = np.expm1(final_pipeline.predict(test))
# df_result = pd.DataFrame(test['Id'])
# df_result['SalePrice'] = result
# df_result.to_csv('submission.csv', index=False)

0:	learn: 0.3496954	total: 17.1ms	remaining: 15.4s
100:	learn: 0.1593947	total: 2.2s	remaining: 17.4s
200:	learn: 0.1287531	total: 4.44s	remaining: 15.4s
300:	learn: 0.1187830	total: 6.45s	remaining: 12.8s
400:	learn: 0.1129202	total: 8.44s	remaining: 10.5s
500:	learn: 0.1094376	total: 10.3s	remaining: 8.19s
600:	learn: 0.1065319	total: 11.9s	remaining: 5.9s
700:	learn: 0.1041605	total: 13.5s	remaining: 3.83s
800:	learn: 0.1021740	total: 15.2s	remaining: 1.88s
899:	learn: 0.1003319	total: 16.9s	remaining: 0us
